# Timing repetitions after the simulation shortlist

The experiment followed a **simulation-first** process:
1. The completed 108-run **no-inference test matrix** compared reuse, capacity and policies cheaply.
2. Those results guided **selected real-inference dev runs**, followed by fixed-block size tuning and full-dev confirmation.
3. This notebook now checks whether the small **timing differences between the shortlisted organizations** repeat. It does **not** repeat the simulation grid or invalidate that selection.

See `docs/no_inference_results.md`, `docs/inference_confirmation.md` and `docs/results.md` for the recorded selection and findings. Simulation establishes cache behavior, not observed GPU TTFT.

## Bounded protocol

- **Qwen2.5-1.5B-Instruct**, pinned model/tokenizer revision, one Colab **T4**.
- **Document / fixed-block-256 / radix**, **LRU**, **4 GiB accelerator FP16**, tensor backend.
- **1,000 queries** per run, random and Zipf, **seed 42**. These are the first 1,000 of each deterministic trace; they are not 1,000 grouped questions.
- **3 fresh-process repetitions** with rotating cache order: document → fixed → radix; fixed → radix → document; radix → document → fixed.
- **One fresh segmented reference per workload**, shared only for correctness checks. No second forward on cache hits, and no repeated controls.
- **18 cached runs + 2 references = 20,000 measured requests**. An 80-request smoke is separate.

Based on previous timings, plan for **roughly 7–9 hours**, not the earlier 19-hour full-trace proposal. There is a persistent **10-hour experiment deadline**, starting with the first smoke/benchmark process. It includes model loading, pauses, retries and checkpoint time after that start. Setup/download time before the smoke is outside this window. Colab does not automatically disconnect when the deadline ends: stop the runtime yourself to stop billing.

If time expires, the current process is killed and its incomplete group is **excluded** from comparisons. A slow T4 is not guaranteed to finish all repetitions within ten hours. No process receives another ten hours when a cell is re-run.

The primary outputs are per-run **mean and p90 TTFT**, plus their median/range across repetitions. The existing full-dev results remain the main accuracy evidence. These 1,000-query repeats test timing stability on selected traces, not full-dev rankings or generalization to other seeds/hardware.

## 1. GPU runtime and repository

Select **Runtime → Change runtime type → T4 GPU**. Upload this notebook to Colab (or open the GitHub copy once it is pushed).

For a **new** suite use `main`. When restoring a checkpoint, use its saved `runtime-lock.json` → `code_revision` below. Do not pull a newer revision halfway through a suite. No Drive mount is required.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_REVISION = "main"  # On restore: exact code_revision from runtime-lock.json.
repo = Path("/content/rag-kvcache")

def checked(*args, **kwargs):
    return subprocess.run(args, check=True, text=True, **kwargs)

checked("nvidia-smi")
if not repo.exists():
    checked("git", "clone", "https://github.com/i0nut02/rag-kvcache.git", str(repo))
os.chdir(repo)
checked("git", "diff", "--exit-code")
checked("git", "diff", "--cached", "--exit-code")
checked("git", "fetch", "origin")
revision = "origin/main" if REPO_REVISION == "main" else REPO_REVISION
checked("git", "checkout", "--detach", revision)
assert Path("configs/strategy_repetitions.json").is_file(), "This revision does not yet contain the new notebook/config."
print("Code revision:", checked("git", "rev-parse", "HEAD", capture_output=True).stdout.strip())

## 2. Install without replacing Colab's CUDA PyTorch stack

Run setup once in a fresh runtime. Do not upgrade packages between repetitions. The notebook records Python, packages and GPU/driver identity and refuses a resume with a changed environment.

If a replacement runtime has different versions, recreate the recorded compatible environment or start a separate suite. Do not bypass the check or merge incompatible timing runs.

In [ ]:
checked(sys.executable, "-c",
        "import torch; assert torch.cuda.is_available(); "
        "print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda); "
        "print('GPU:', torch.cuda.get_device_name(0))")

# Optional Colab secret; never print the token or write it into a checkpoint.
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if token:
    os.environ["HF_TOKEN"] = token
print("HF token configured:", bool(token))

In [ ]:
import hashlib
import json
import urllib.request

dataset = Path("data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev")
expected_checksum = "99852d874994078e4b4112b71ceca4dd35aa3a24ff6d3a35c051be25295b4fef"
dataset.parent.mkdir(parents=True, exist_ok=True)
if not dataset.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/nyu-mll/quality/main/data/v1.0.1/" + dataset.name,
        dataset,
    )
assert hashlib.sha256(dataset.read_bytes()).hexdigest() == expected_checksum, "Unexpected dataset checksum"
checked(sys.executable, "experiments/run_quality.py", "validate-data", str(dataset),
        "--split", "dev", "--verify-counts")
checked(sys.executable, "-m", "unittest", "tests.test_strategy_repetitions",
        "tests.test_reference", "tests.test_radix_access", "-q")

## 3. Restore only if continuing this bounded experiment

The notebook always uses the **confirmation profile: 1,000 queries**. There is no full-trace profile in this new config, preventing an accidental 19-hour run.

Checkpoints contain only this suite's results, manifests, receipts, config, environment record and deadline—no dataset or model weights. Download them to your computer, **without Google Drive**. A completed run can be skipped; an interrupted run must restart at request zero.

In a replacement runtime, upload your latest ZIP **before** freezing the environment below. The saved ten-hour deadline still applies, including downtime. A checkpoint restored after the deadline remains analyzable but cannot start more inference. Do not delete the deadline to silently extend the agreed budget.

In [ ]:
from google.colab import files
import zipfile

PROFILE = "confirmation"  # Fixed 1,000-query timing protocol.
suite_root = Path("results/strategy_repetitions")
suite_root.mkdir(parents=True, exist_ok=True)

def restore_checkpoint():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one strategy-repetitions checkpoint ZIP")
    archive = Path(next(iter(uploaded)))
    with zipfile.ZipFile(archive) as bundle:
        if "runtime-lock.json" not in bundle.namelist():
            raise ValueError("Not a checkpoint from this notebook")
        # Refuse path traversal, symlinks and overwriting different local evidence.
        for member in bundle.infolist():
            target = (suite_root / member.filename).resolve()
            if not target.is_relative_to(suite_root.resolve()):
                raise ValueError("Unsafe archive path")
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Archive contains a symlink")
            if target.is_file() and target.read_bytes() != bundle.read(member):
                raise ValueError(f"Different local artifact exists: {target}")
        bundle.extractall(suite_root)
    print("Restored:", archive)
    del uploaded  # Release the upload bytes.

# Uncomment only when restoring this notebook's checkpoint.
# restore_checkpoint()

## 4. Freeze the protocol and inspect the schedule

Each run starts in a **new Python process with an empty article cache**. Model loading and pinned-L0 setup are outside the existing request TTFT timer, but count against the ten-hour experiment deadline. The smoke does not warm a later timed process.

The full 1,000-request timing is primary and includes cold-start behavior. A separately labelled sensitivity view drops the first **100** requests without resetting the cache. This is not a claim of stationary cache behavior.

Do not change packages, source or trace length between repetitions. The execution receipt records runtime session boundaries. The references are run only once and must not be presented as three independent uncached timing measurements.

In [ ]:
import importlib.metadata
import platform
import time
import uuid
import pandas as pd

from src.quality_cache.matrix import load_matrix, build_matrix_commands
from src.quality_cache.experiment_budget import ExperimentBudget, BudgetExpired
from src.quality_cache.reporting.repetitions import (
    analyze_repetitions, artifact_paths, read_repetition_run, sha256,
)

config_path = Path("configs/strategy_repetitions.json")
config = load_matrix(config_path)
assert PROFILE == "confirmation"
output_dir = suite_root / PROFILE
output_dir.mkdir(exist_ok=True)

def environment_lock():
    source_files = checked("git", "ls-files", "src", "experiments", "configs",
                            capture_output=True).stdout.splitlines()
    source_digest = hashlib.sha256()
    for name in sorted(source_files):
        source_digest.update(name.encode() + b"\0" + Path(name).read_bytes())
    packages = {}
    for name in ("torch", "transformers", "accelerate", "numpy", "tokenizers",
                 "huggingface-hub", "safetensors", "psutil", "triton"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    gpu = checked("nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                  "--format=csv,noheader,nounits", capture_output=True).stdout.strip()
    if len(gpu.splitlines()) != 1 or "T4" not in gpu:
        raise ValueError("This protocol requires one T4. Use a separate protocol for a different GPU.")
    return {
        "protocol": "quality-strategy-repetitions-v1",
        "profile": PROFILE,
        "code_revision": checked("git", "rev-parse", "HEAD", capture_output=True).stdout.strip(),
        "source_sha256": source_digest.hexdigest(),
        "matrix_sha256": sha256(config_path),
        "dataset_sha256": sha256(dataset),
        "python": platform.python_version(),
        "packages": packages, "gpu_driver_memory": gpu,
        "allocator": os.environ["PYTORCH_CUDA_ALLOC_CONF"],
    }

lock = environment_lock()
lock_path = suite_root / "runtime-lock.json"
if lock_path.exists():
    if json.loads(lock_path.read_text()) != lock:
        raise ValueError("Checkpoint environment/protocol differs. Restore the recorded revision/versions, or use a new suite directory.")
else:
    if list(output_dir.glob("*.jsonl")):
        raise ValueError("Untracked results exist without an environment lock. Do not import historical rows.")
    lock_path.write_text(json.dumps(lock, indent=2) + "\n")
    (suite_root / "protocol.json").write_text(config_path.read_text())
    (suite_root / "pip-freeze.txt").write_text(
        checked(sys.executable, "-m", "pip", "freeze", capture_output=True).stdout
    )
session_id = str(uuid.uuid4())  # Informational: a restart may use another physical T4.
print(json.dumps(lock, indent=2))
budget = ExperimentBudget(suite_root / "time-budget.json", limit_s=config["benchmark_limit_hours"] * 3600)
print(f"Experiment window remaining: {budget.remaining() / 3600:.2f} hours (starts at first smoke)")
schedule = pd.DataFrame(config["runs"])[["group", "name", "repetition", "workload", "policy", "cache_strategy"]]
display(schedule)

## 5. Execution and checkpoint helpers

A receipt is saved only after the JSONL, summary and manifest validate. Resume verifies their hashes, exact command and environment lock; a JSONL alone is **not** a completion marker. Incomplete artifacts are preserved before restarting a run.

The watchdog kills a benchmark at the saved deadline **even if no progress is printed**. A timeout saves a local ZIP and stops the cell. Previously completed runs stay available. Click the notebook's download cell or use Colab's Files panel to retrieve `/content/strategy-repetitions-latest.zip`.

Each finished group downloads a checkpoint. The ZIP is also refreshed after each completed run. Browser downloads do not use Drive storage. Pauses while downloading count against the ten-hour window; avoid long breaks if you want all three repetitions.

In [ ]:
import shutil
from datetime import datetime, timezone

def checkpoint_zip():
    return shutil.make_archive("/content/strategy-repetitions-latest", "zip", root_dir=suite_root)

def download_checkpoint():
    archive = checkpoint_zip()
    print(f"Checkpoint: {Path(archive).stat().st_size / 2**20:.1f} MiB")
    files.download(archive)

def verify_completed(spec, command, profile):
    output = Path(command[command.index("--output") + 1])
    receipt_path = output.with_suffix(".execution.json")
    if not receipt_path.exists():
        return False
    receipt = json.loads(receipt_path.read_text())
    stamp = {"command": command, "lock_sha256": sha256(lock_path)}
    if any(receipt.get(k) != v for k, v in stamp.items()):
        raise ValueError(f"Resume command/lock mismatch: {output.name}")
    read_repetition_run(output, spec, config, profile)
    for path in artifact_paths(output):
        if receipt["sha256"].get(path.name) != sha256(path):
            raise ValueError(f"Changed completed artifact: {path}")
    return True

def run_one(spec, command, profile):
    if environment_lock() != lock:
        raise ValueError("Code or environment changed since the suite was frozen.")
    output = Path(command[command.index("--output") + 1])
    output.parent.mkdir(parents=True, exist_ok=True)
    if verify_completed(spec, command, profile):
        print("Skip verified completed run:", output.name, flush=True)
        return
    leftovers = [p for p in artifact_paths(output) if p.exists()]
    if leftovers:
        backup = output.parent / "incomplete" / (output.stem + "-" + uuid.uuid4().hex[:8])
        backup.mkdir(parents=True)
        for path in leftovers:
            shutil.move(str(path), backup / path.name)
        print("Preserved incomplete artifacts:", backup, flush=True)
    started = datetime.now(timezone.utc).isoformat()
    try:
        wall_time = budget.run(
            [sys.executable, "-u", "experiments/run_quality.py", *command],
            log_path=output.with_suffix(".console.log"),
        )
    except (BudgetExpired, KeyboardInterrupt, subprocess.CalledProcessError):
        checkpoint_zip()
        print("Stopped. Completed runs are safe; download the latest ZIP before ending the runtime.")
        raise
    read_repetition_run(output, spec, config, profile)
    receipt = {
        "command": command, "lock_sha256": sha256(lock_path),
        "session_id": session_id, "started_at": started, "wall_time_s": wall_time,
        "sha256": {p.name: sha256(p) for p in artifact_paths(output)},
        "gpu_state_after": checked("nvidia-smi", capture_output=True).stdout,
    }
    receipt_path = output.with_suffix(".execution.json")
    temporary = receipt_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(receipt, indent=2) + "\n")
    temporary.replace(receipt_path)
    checkpoint_zip()

def run_group(group, profile=PROFILE):
    if group not in range(1, 7):
        raise ValueError("Choose group 1..6")
    print(f"Time window remaining: {budget.remaining() / 3600:.2f} hours")
    commands = build_matrix_commands(config, profile, suite_root / profile)
    for spec, command in zip(config["runs"], commands):
        if spec["group"] == group:
            print(f"Group {group}/6: {spec['name']} ({profile})", flush=True)
            run_one(spec, command, profile)
    if profile == PROFILE:
        download_checkpoint()

## 6. Ten-request smoke (not timing evidence)

Exercise all three caches plus a reference on each workload, just once: **80 requests**. Smoke output is separate and excluded from timing analysis. **The ten-hour deadline starts here** and counts any first-time weight download/model loading. Pre-download the pinned model if you want to keep that download outside the benchmark window; otherwise simply allow time for it.

The selection was already made using simulation and inference evidence. This smoke is only a functional gate, not another strategy-selection stage.

In [ ]:
run_group(1, profile="smoke")
run_group(2, profile="smoke")

## 7. Timing measurements — six checkpoints within the same deadline

Each run serves exactly **1,000 queries**. Groups 1 and 2 contain a reference plus the three caches. Groups 3–6 contain only the three caches, using their workload's saved reference.

Run cells in order, or use Colab's Run all. Groups take roughly 1–2 hours depending on workload and GPU speed. Checkpoint after each group. No inference starts past the saved deadline, and an active benchmark is killed when it reaches that deadline.

If a timeout occurs, skip to analysis and set `ALLOW_INCOMPLETE = True`. It reports only complete three-strategy groups with their **actual repetition counts**; it never labels an interrupted run a completed 1,000-query measurement.

### Group 1 — repetition 1, random
Reference (once) → document → fixed-block → radix.

In [ ]:
run_group(1)

### Group 2 — repetition 1, Zipf
Reference (once) → document → fixed-block → radix.

In [ ]:
run_group(2)

### Group 3 — repetition 2, Zipf
Fixed-block → radix → document; reuse the saved Zipf correctness reference.

In [ ]:
run_group(3)

### Group 4 — repetition 2, random
Fixed-block → radix → document; reuse the saved random correctness reference.

In [ ]:
run_group(4)

### Group 5 — repetition 3, random
Radix → document → fixed-block; reuse the saved random correctness reference.

In [ ]:
run_group(5)

### Group 6 — repetition 3, Zipf
Radix → document → fixed-block; reuse the saved Zipf correctness reference.

In [ ]:
run_group(6)

## 8. Validate and compare complete repetitions

By default this expects all **20 runs**. After a deadline stop, `ALLOW_INCOMPLETE = True` can analyze completed groups. Never pool half-finished runs or count only the faster strategies of an interrupted group.

The analyzer rejects mixed schemas, model/code/hardware provenance, changed references, truncated completed runs and misaligned traces. Zipf alignment includes trace position, not just Q&A ID.

The two controls appear only once (repetition 0) for correctness context. **Paired timing comparisons are between cached strategies**, within repetition and workload. There is no repeated-control speedup claim and no IID request bootstrap. Report ranges of per-run mean and p90, not a pooled percentile or a confidence interval.

In [ ]:
ALLOW_INCOMPLETE = False  # Set True after a deadline stop to inspect complete groups only.

# Verify receipts without launching any model process.
commands = build_matrix_commands(config, PROFILE, output_dir)
validated_specs = []
missing = []
for spec, command in zip(config["runs"], commands):
    if verify_completed(spec, command, PROFILE):
        validated_specs.append(spec)
    else:
        missing.append(spec["name"])
if missing and not ALLOW_INCOMPLETE:
    raise ValueError(f"Missing runs: {missing}. Finish within the deadline or enable partial analysis.")
print("Missing/unreceipted runs:", missing)

# Ignore artifacts without completion receipts, even if their filenames exist.
analysis_config = {**config, "runs": validated_specs}
analysis_dir = suite_root / f"analysis-{PROFILE}"
artifacts = analyze_repetitions(
    analysis_config, output_dir, analysis_dir, profile=PROFILE, allow_incomplete=ALLOW_INCOMPLETE,
)
coverage_path = analysis_dir / "coverage.json"
coverage_path.write_text(json.dumps({
    "planned_runs": len(config["runs"]), "completed_runs": len(validated_specs),
    "missing_runs": missing, "time_budget": json.loads(budget.path.read_text()),
}, indent=2) + "\n")
runs = pd.DataFrame(artifacts["run_metrics"])
variation = pd.DataFrame(artifacts["strategy_variation"])
pairs = pd.DataFrame(artifacts["paired_variation"])

display(runs.query("scope == 'all_requests'")[[
    "workload", "repetition", "strategy", "requests", "ttft_mean_s", "ttft_p90_s",
    "article_token_hit_rate", "policy_mean_s", "lookup_mean_s",
    "accuracy", "reference_label_agreement", "reference_label_mismatches",
]])
display(variation.query("scope == 'all_requests' and strategy != 'segmented'")[[
    "workload", "strategy", "repetitions",
    "ttft_mean_s_median", "ttft_mean_s_min", "ttft_mean_s_max", "ttft_mean_s_stdev",
    "ttft_p90_s_median", "ttft_p90_s_min", "ttft_p90_s_max",
]])
display(pairs.query("scope == 'all_requests'"))
# Positive reduction means the candidate is faster than the baseline cache.
# Optional diagnostic view:
# display(pairs.query("scope == 'after_first_10_percent'"))

In [ ]:
# Plot individual run means and p90s, not thousands of pseudo-replicated requests.
import matplotlib.pyplot as plt

primary = runs.query("scope == 'all_requests' and strategy != 'segmented'")
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
for row, workload in enumerate(("random", "zipf")):
    for col, metric in enumerate(("ttft_mean_s", "ttft_p90_s")):
        ax = axes[row, col]
        for strategy, selected in primary.query("workload == @workload").groupby("strategy"):
            selected = selected.sort_values("repetition")
            ax.plot(selected.repetition, selected[metric], "o-", label=strategy)
        ax.set(title=f"{workload}: {metric}", xlabel="Fresh-process repetition", ylabel="seconds")
        ax.set_xticks([1, 2, 3])
        ax.legend()
fig.savefig(analysis_dir / "run_variation.png", dpi=160)
plt.show()
download_checkpoint()

## 9. How to describe the evidence

1. Explain the sequence: **broad no-inference test matrix → selected dev inference → targeted timing repetitions**. The simulation justified the shortlist through reuse, capacity and management behavior; only inference measures TTFT.
2. Keep LRU fixed to isolate organization. This does not claim LRU beat GDSF on every Zipf case. Grouped locality, LFU, other budgets, CPU INT8, arena and Triton already have separate evidence and are not re-swept here.
3. Report **up to three 1,000-query repetitions**, their actual counts, median and min–max mean TTFT, and median/range of p90. Paired comparisons use complete groups only. Disclose a deadline-truncated study.
4. If the fastest cache changes between repetitions, say the ordering is unstable. Even 3/3 wins is limited evidence, not proof of a tiny effect or a universal ranking.
5. Investigate changed hit rates or label mismatches; do not attribute every timing difference to the data structure. Show the first-10% exclusion only as a diagnostic, never as cherry-picked replacement evidence.
6. References are **single correctness controls**, not repeated timing controls. Cached strategies are compared directly within each repetition.
7. Keep this current revision/schema separate from historical full-dev tables, especially because radix access accounting changed. The 1,000-query subset accuracy is not full-dev accuracy; Zipf also repeats Q&As.
8. The new runs test **timing repeatability of selected traces on T4**, not new seeds, other GPUs, production vLLM/SGLang engines or serving concurrency.

Save the final ZIP (raw outputs, manifests, receipts, configuration, environment/deadline, CSVs and plots). **Disconnect/delete the Colab runtime when done**; the process timeout is not a Colab billing shutdown. No new experimental conclusion exists until these runs finish.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/strategy-repetitions-latest",
    "zip",
    root_dir="results/strategy_repetitions",
)
print("Downloading:", archive)
files.download(archive)

In [ ]:
import time
from datetime import datetime

duration_s = 2 * 60 * 60
end = time.monotonic() + duration_s

while time.monotonic() < end:
    remaining_min = (end - time.monotonic()) / 60
    print(
        datetime.now().strftime("%H:%M:%S"),
        f"— approximately {remaining_min:.0f} minutes remaining",
        flush=True,
    )
    time.sleep(min(300, max(0, end - time.monotonic())))

print("Two-hour waiting period completed.")

In [ ]:
import time
from datetime import datetime

duration_s = 2 * 60 * 60
end = time.monotonic() + duration_s

while time.monotonic() < end:
    remaining_min = (end - time.monotonic()) / 60
    print(
        datetime.now().strftime("%H:%M:%S"),
        f"— approximately {remaining_min:.0f} minutes remaining",
        flush=True,
    )
    time.sleep(min(300, max(0, end - time.monotonic())))

print("Two-hour waiting period completed.")